<a href="https://colab.research.google.com/github/purnimakushwaha/ITC101_Minor-project_python/blob/main/Duplicate_file_finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#          DUPLICATE FILE FINDER & STORAGE ANALYZER
# ============================================================

import os
import hashlib
import pandas as pd
import time
from datetime import datetime


# ============================================================
# SETTINGS
# ============================================================

REPORT_FILE = "duplicate_files_report.csv"


# ============================================================
# CALCULATE FILE HASH
# ============================================================

def calculate_hash(file_path, chunk_size=1024 * 1024):
    """
    Calculates SHA-256 hash of a file.
    File is read in chunks so large files can also be handled.
    """

    sha256 = hashlib.sha256()

    try:

        with open(file_path, "rb") as file:

            while True:

                data = file.read(chunk_size)

                if not data:
                    break

                sha256.update(data)

        return sha256.hexdigest()

    except (PermissionError, OSError):

        return None


# ============================================================
# GET FILE SIZE
# ============================================================

def get_file_size(file_path):

    try:

        return os.path.getsize(file_path)

    except (PermissionError, OSError):

        return 0


# ============================================================
# CONVERT BYTES TO READABLE FORMAT
# ============================================================

def format_size(size):

    if size < 1024:
        return f"{size} B"

    elif size < 1024 ** 2:
        return f"{size / 1024:.2f} KB"

    elif size < 1024 ** 3:
        return f"{size / (1024 ** 2):.2f} MB"

    elif size < 1024 ** 4:
        return f"{size / (1024 ** 3):.2f} GB"

    else:
        return f"{size / (1024 ** 4):.2f} TB"


# ============================================================
# SCAN FOLDER
# ============================================================

def scan_folder(folder_path):

    print("\n" + "=" * 70)
    print("                    SCANNING FOLDER")
    print("=" * 70)

    print("Folder:", folder_path)

    print("\nPlease wait...\n")

    start_time = time.perf_counter()

    files = []

    for root, directories, filenames in os.walk(folder_path):

        for filename in filenames:

            file_path = os.path.join(
                root,
                filename
            )

            try:

                file_size = os.path.getsize(
                    file_path
                )

                files.append({
                    "path": file_path,
                    "name": filename,
                    "size": file_size
                })

            except (PermissionError, OSError):

                continue

    end_time = time.perf_counter()

    scan_time = end_time - start_time

    print(
        "Files found:",
        len(files)
    )

    print(
        f"Folder scan time: {scan_time:.6f} seconds"
    )

    return files


# ============================================================
# FIND DUPLICATES
# ============================================================

def find_duplicates(files):

    print("\n" + "=" * 70)
    print("                 FINDING DUPLICATES")
    print("=" * 70)

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # STEP 1:
    # Group files by size
    # --------------------------------------------------------

    size_groups = {}

    for file in files:

        size = file["size"]

        if size not in size_groups:

            size_groups[size] = []

        size_groups[size].append(file)


    # --------------------------------------------------------
    # STEP 2:
    # Only files with same size can potentially be duplicates
    # --------------------------------------------------------

    possible_duplicates = []

    for size, group in size_groups.items():

        if len(group) > 1:

            possible_duplicates.extend(group)


    print(
        "\nFiles with matching sizes:",
        len(possible_duplicates)
    )


    # --------------------------------------------------------
    # STEP 3:
    # Calculate hash only for possible duplicates
    # --------------------------------------------------------

    hash_groups = {}

    for index, file in enumerate(
        possible_duplicates,
        start=1
    ):

        file_hash = calculate_hash(
            file["path"]
        )

        if file_hash is None:

            continue

        file["hash"] = file_hash

        if file_hash not in hash_groups:

            hash_groups[file_hash] = []

        hash_groups[file_hash].append(
            file
        )


    # --------------------------------------------------------
    # STEP 4:
    # Keep only groups containing more than one file
    # --------------------------------------------------------

    duplicates = {

        file_hash: group

        for file_hash, group
        in hash_groups.items()

        if len(group) > 1
    }


    end_time = time.perf_counter()

    hash_time = end_time - start_time

    print(
        "Duplicate groups found:",
        len(duplicates)
    )

    print(
        f"Hashing & comparison time: "
        f"{hash_time:.6f} seconds"
    )

    return duplicates


# ============================================================
# DISPLAY DUPLICATES
# ============================================================

def display_duplicates(duplicates):

    print("\n" + "=" * 70)
    print("                    DUPLICATE FILES")
    print("=" * 70)

    if not duplicates:

        print(
            "\nNo duplicate files found!"
        )

        return


    group_number = 1

    duplicate_file_count = 0

    wasted_space = 0


    for file_hash, group in duplicates.items():

        print("\n" + "-" * 70)

        print(
            f"DUPLICATE GROUP {group_number}"
        )

        print("-" * 70)

        print(
            "Hash:",
            file_hash
        )

        print(
            "File size:",
            format_size(
                group[0]["size"]
            )
        )

        print()


        for index, file in enumerate(
            group,
            start=1
        ):

            print(
                f"{index}. {file['name']}"
            )

            print(
                f"   Path: {file['path']}"
            )

            duplicate_file_count += 1


        # First copy is considered original.
        # Remaining copies represent wasted space.

        wasted_space += (
            group[0]["size"]
            * (len(group) - 1)
        )


        group_number += 1


    print("\n" + "=" * 70)

    print(
        "Duplicate files:",
        duplicate_file_count
    )

    print(
        "Potential wasted space:",
        format_size(
            wasted_space
        )
    )

    print("=" * 70)


# ============================================================
# GENERATE REPORT
# ============================================================

def generate_report(duplicates):

    if not duplicates:

        print(
            "\nNo duplicate data available for report."
        )

        return


    report = []

    group_number = 1


    for file_hash, group in duplicates.items():

        original_file = group[0]["path"]

        for index, file in enumerate(
            group
        ):

            if index == 0:

                status = "Original"

                wasted_space = 0

            else:

                status = "Duplicate"

                wasted_space = file["size"]


            report.append({

                "Group":
                    group_number,

                "File Name":
                    file["name"],

                "File Path":
                    file["path"],

                "File Size (Bytes)":
                    file["size"],

                "File Size":
                    format_size(
                        file["size"]
                    ),

                "SHA-256":
                    file_hash,

                "Status":
                    status,

                "Potential Wasted Space":
                    format_size(
                        wasted_space
                    )
            })


        group_number += 1


    df = pd.DataFrame(report)


    df.to_csv(
        REPORT_FILE,
        index=False
    )


    print("\n" + "=" * 70)

    print(
        "                    REPORT GENERATED"
    )

    print("=" * 70)

    print(
        "Report saved as:",
        REPORT_FILE
    )

    print(
        "Total report records:",
        len(df)
    )


# ============================================================
# SHOW STATISTICS
# ============================================================

def show_statistics(files, duplicates, execution_time):

    print("\n" + "=" * 70)

    print(
        "                  STORAGE STATISTICS"
    )

    print("=" * 70)


    total_files = len(files)

    total_size = sum(
        file["size"]
        for file in files
    )


    duplicate_count = 0

    wasted_space = 0


    for group in duplicates.values():

        duplicate_count += (
            len(group) - 1
        )

        wasted_space += (
            group[0]["size"]
            * (len(group) - 1)
        )


    unique_files = (
        total_files
        - duplicate_count
    )


    print(
        f"\nTotal files scanned       : "
        f"{total_files}"
    )

    print(
        f"Unique files              : "
        f"{unique_files}"
    )

    print(
        f"Duplicate files           : "
        f"{duplicate_count}"
    )

    print(
        f"Duplicate groups          : "
        f"{len(duplicates)}"
    )

    print(
        f"Total storage scanned     : "
        f"{format_size(total_size)}"
    )

    print(
        f"Potential wasted storage  : "
        f"{format_size(wasted_space)}"
    )

    print(
        f"Total execution time      : "
        f"{execution_time:.6f} seconds"
    )


    if total_size > 0:

        percentage = (
            wasted_space
            / total_size
        ) * 100

        print(
            f"Potential wasted percentage: "
            f"{percentage:.2f}%"
        )


    print("=" * 70)


# ============================================================
# SEARCH DUPLICATE FILE
# ============================================================

def search_duplicates(duplicates):

    print("\n" + "=" * 70)

    print(
        "                  SEARCH DUPLICATES"
    )

    print("=" * 70)


    if not duplicates:

        print(
            "No duplicate files available."
        )

        return


    keyword = input(
        "Enter file name or keyword: "
    ).strip().lower()


    if keyword == "":

        print(
            "Please enter a keyword."
        )

        return


    found = []


    for group in duplicates.values():

        for file in group:

            if keyword in file["name"].lower():

                found.append(file)


    if not found:

        print(
            "\nNo matching duplicate file found."
        )

        return


    print(
        f"\n{len(found)} matching file(s) found:"
    )


    for index, file in enumerate(
        found,
        start=1
    ):

        print(
            f"\n{index}. {file['name']}"
        )

        print(
            "   Path:",
            file["path"]
        )

        print(
            "   Size:",
            format_size(
                file["size"]
            )
        )


# ============================================================
# VIEW REPORT
# ============================================================

def view_report():

    print("\n" + "=" * 70)

    print(
        "                    SAVED REPORT"
    )

    print("=" * 70)


    if not os.path.exists(
        REPORT_FILE
    ):

        print(
            "No saved report found."
        )

        return


    try:

        df = pd.read_csv(
            REPORT_FILE
        )


        if df.empty:

            print(
                "Report is empty."
            )

            return


        print(
            df.to_string(
                index=False
            )
        )


    except Exception as error:

        print(
            "Unable to read report:",
            error
        )


# ============================================================
# GET FOLDER FROM USER
# ============================================================

def get_folder():

    print("\n" + "=" * 70)

    print(
        "                     FOLDER SELECTION"
    )

    print("=" * 70)

    print(
        "\nEnter the complete folder path."
    )

    print(
        r"Example: D:\Purnima\Downloads"
    )

    print(
        "You can also enter '.' for the current "
        "Jupyter working folder."
    )


    folder_path = input(
        "\nFolder path: "
    ).strip()


    # Remove quotation marks if user pasted
    # a path surrounded by quotes.

    folder_path = folder_path.strip(
        '"'
    ).strip("'"
    )


    if folder_path == "":

        print(
            "Folder path cannot be empty."
        )

        return None


    if folder_path == ".":

        folder_path = os.getcwd()


    if not os.path.isdir(
        folder_path
    ):

        print(
            "\nFolder does not exist!"
        )

        return None


    return os.path.abspath(
        folder_path
    )


# ============================================================
# COMPLETE SCAN
# ============================================================

def start_scan():

    folder_path = get_folder()


    if folder_path is None:

        return


    total_start = time.perf_counter()


    files = scan_folder(
        folder_path
    )


    if len(files) == 0:

        print(
            "\nNo files found in this folder."
        )

        return


    duplicates = find_duplicates(
        files
    )


    total_end = time.perf_counter()

    total_execution_time = (
        total_end - total_start
    )


    display_duplicates(
        duplicates
    )


    show_statistics(
        files,
        duplicates,
        total_execution_time
    )


    # Ask whether user wants report

    save_choice = input(
        "\nGenerate CSV report? (y/n): "
    ).strip().lower()


    if save_choice == "y":

        generate_report(
            duplicates
        )


# ============================================================
# DEMO / TEST DATA EXPLANATION
# ============================================================

def show_project_info():

    print("\n" + "=" * 70)

    print(
        "                 ABOUT THIS PROJECT"
    )

    print("=" * 70)

    print(
        "\nProject Name:"
    )

    print(
        "Duplicate File Finder & Storage Analyzer"
    )


    print(
        "\nPurpose:"
    )

    print(
        "Finds files with identical content inside "
        "a selected folder."
    )


    print(
        "\nHow it works:"
    )

    print(
        "1. Scans files"
    )

    print(
        "2. Groups files by size"
    )

    print(
        "3. Calculates SHA-256 hash"
    )

    print(
        "4. Compares file hashes"
    )

    print(
        "5. Finds duplicate groups"
    )

    print(
        "6. Calculates wasted storage"
    )


    print(
        "\nImportant:"
    )

    print(
        "The program does NOT automatically delete files."
    )

    print("=" * 70)


# ============================================================
# MAIN MENU
# ============================================================

def main_menu():

    while True:

        print("\n")

        print("=" * 70)

        print(
            "          DUPLICATE FILE FINDER & STORAGE ANALYZER"
        )

        print("=" * 70)

        print("1.  Scan Folder")

        print("2.  Search Duplicate Files")

        print("3.  View Saved CSV Report")

        print("4.  Project Information")

        print("5.  Exit")

        print("=" * 70)


        choice = input(
            "Enter your choice: "
        ).strip()


        if choice == "1":

            # Start folder scanning

            start_scan()


        elif choice == "2":

            # Search requires a fresh scan

            print(
                "\nTo search duplicates, first scan a folder."
            )

            folder_path = get_folder()


            if folder_path is not None:

                files = scan_folder(
                    folder_path
                )

                duplicates = find_duplicates(
                    files
                )

                search_duplicates(
                    duplicates
                )


        elif choice == "3":

            view_report()


        elif choice == "4":

            show_project_info()


        elif choice == "5":

            print(
                "\n" + "=" * 70
            )

            print(
                "Thank you for using "
                "Duplicate File Finder!"
            )

            print(
                "Keep learning Python! 🐍"
            )

            print(
                "=" * 70
            )

            break


        else:

            print(
                "\nInvalid choice!"
            )

            print(
                "Please select a number from 1 to 5."
            )


# ============================================================
# START PROJECT
# ============================================================

print("=" * 70)

print(
    "          DUPLICATE FILE FINDER & STORAGE ANALYZER"
)


print("=" * 70)

now = datetime.now()

print(
    "Date:",
    now.strftime("%d-%m-%Y")
)

print(
    "Time:",
    now.strftime("%I:%M:%S %p")
)

print("=" * 70)

print(
    "\nProject is ready!"
)

main_menu()

          DUPLICATE FILE FINDER & STORAGE ANALYZER
Date: 10-08-2026
Time: 02:52:49 PM

Project is ready!


          DUPLICATE FILE FINDER & STORAGE ANALYZER
1.  Scan Folder
2.  Search Duplicate Files
3.  View Saved CSV Report
4.  Project Information
5.  Exit
Enter your choice: 1

                     FOLDER SELECTION

Enter the complete folder path.
Example: D:\Purnima\Downloads
You can also enter '.' for the current Jupyter working folder.

Folder path: .

                    SCANNING FOLDER
Folder: /content

Please wait...

Files found: 21
Folder scan time: 0.003440 seconds

                 FINDING DUPLICATES

Files with matching sizes: 4
Duplicate groups found: 0
Hashing & comparison time: 0.002297 seconds

                    DUPLICATE FILES

No duplicate files found!

                  STORAGE STATISTICS

Total files scanned       : 21
Unique files              : 21
Duplicate files           : 0
Duplicate groups          : 0
Total storage scanned     : 54.28 MB
Potential wasted 